In [1]:
import os

# --- FIX: Prevent TF/protobuf crash (MessageFactory.GetPrototype) by forcing pure-Python protobuf
# This must run before *any* TensorFlow import.
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION"] = "3"
os.environ["TF_PROTOBUF_IMPLEMENTATION"] = "python"

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

import random
import numpy as np
import pandas as pd

# --- FIX: Make TF import robust; if it still fails, we'll fall back to a safe submission later.
TF_AVAILABLE = True
try:
    import tensorflow as tf
    from tensorflow.keras import layers
except Exception as e:
    TF_AVAILABLE = False
    TF_IMPORT_ERROR = repr(e)
    tf = None
    layers = None
    print("WARNING: TensorFlow failed to import; will create fallback submission.")
    print("TF import error:", TF_IMPORT_ERROR)

try:
    from skimage.transform import resize as _sk_resize

    def resize(img, out_shape, preserve_range=True, anti_aliasing=True):
        return _sk_resize(
            img,
            out_shape,
            preserve_range=preserve_range,
            anti_aliasing=anti_aliasing,
        )

except Exception:
    # NOTE: This path requires TensorFlow. If TF isn't available, we won't call resize().
    def resize(img, out_shape, preserve_range=True, anti_aliasing=True):
        x = tf.convert_to_tensor(img, dtype=tf.float32)
        x = x[tf.newaxis, ..., tf.newaxis]  # (1,H,W,1)
        x = tf.image.resize(
            x, out_shape, method="bilinear", antialias=bool(anti_aliasing)
        )
        x = tf.squeeze(x, axis=(0, 3))
        return x.numpy()


random.seed(SEED)
np.random.seed(SEED)
if TF_AVAILABLE:
    tf.random.set_seed(SEED)

INPUT_ROOT = "/kaggle/input/rsna-miccai-brain-tumor-radiogenomic-classification"
TRAIN_DIR = os.path.join(INPUT_ROOT, "train")
TEST_DIR = os.path.join(INPUT_ROOT, "test")
LABELS_CSV = os.path.join(INPUT_ROOT, "train_labels.csv")
SAMPLE_SUB = os.path.join(INPUT_ROOT, "sample_submission.csv")

IMG_PX_SIZE = 150
SLICES_PER_CASE = 6
BAD_CASES = {"00109", "00123", "00709"}

print("Using TRAIN_DIR:", TRAIN_DIR)
print("Using TEST_DIR:", TEST_DIR)
print("Using LABELS_CSV:", LABELS_CSV)


def _safe_dcm_read(path):
    """
    Returns a 2D float32 numpy array or None.
    Uses tf.io.decode_dicom_image to avoid pydicom runtime issues.
    """
    if not TF_AVAILABLE:
        return None
    try:
        b = tf.io.read_file(path)
        img = tf.io.decode_dicom_image(
            b,
            dtype=tf.uint16,
            color_dim=False,
            scale="preserve",
            expand_animations=True,
        )
        img = tf.squeeze(img)

        try:
            if getattr(img, "shape", None) is not None and img.shape.rank == 3:
                img = img[0]
        except Exception:
            pass

        arr = img.numpy().astype(np.float32)
        if arr.ndim != 2:
            return None
        return arr
    except Exception:
        return None




2026-05-13 01:16:12.880974: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-13 01:16:12.949143: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778634972.984928      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778634972.995314      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-13 01:16:13.055887: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Using TRAIN_DIR: /kaggle/input/rsna-miccai-brain-tumor-radiogenomic-classification/train
Using TEST_DIR: /kaggle/input/rsna-miccai-brain-tumor-radiogenomic-classification/test
Using LABELS_CSV: /kaggle/input/rsna-miccai-brain-tumor-radiogenomic-classification/train_labels.csv


In [2]:
def _normalize_img(img2d):
    img2d = img2d.astype(np.float32)
    mn = np.min(img2d)
    mx = np.max(img2d)
    if mx - mn < 1e-6:
        return None
    img2d = (img2d - mn) / (mx - mn)
    return img2d


def _load_case_t2_slices(
    case_dir, img_px_size=150, slices_per_case=6, sum_thr=90000.0, norm_sum_thr=3000.0
):
    """
    Core logic preserved: scan T2w DICOMs, pick slices passing thresholds,
    resize to IMG_PX_SIZE, stack to 3 channels, normalize.
    Returns (slices_per_case, img_px_size, img_px_size, 3) float32.
    If not enough slices, pads by repeating last valid (or zeros if none).
    """
    # If TF isn't available, return zeros so downstream code can still run if needed.
    if not TF_AVAILABLE:
        return np.zeros(
            (slices_per_case, img_px_size, img_px_size, 3), dtype=np.float32
        )

    subdirs = sorted([f.path for f in os.scandir(case_dir) if f.is_dir()])
    t2_dir = None
    for d in subdirs:
        if os.path.basename(d).lower() == "t2w":
            t2_dir = d
            break
    if t2_dir is None:
        if len(subdirs) > 0:
            t2_dir = subdirs[-1]
        else:
            out = np.zeros(
                (slices_per_case, img_px_size, img_px_size, 3), dtype=np.float32
            )
            return out

    dcm_files = sorted(
        [
            f.path
            for f in os.scandir(t2_dir)
            if f.is_file() and f.name.lower().endswith(".dcm")
        ]
    )
    collected = []
    for fp in dcm_files:
        arr = _safe_dcm_read(fp)
        if arr is None:
            continue
        if float(np.sum(arr)) <= sum_thr:
            continue
        arr = resize(
            arr, (img_px_size, img_px_size), preserve_range=True, anti_aliasing=True
        ).astype(np.float32)
        arr = _normalize_img(arr)
        if arr is None:
            continue
        stacked = np.stack((arr, arr, arr), axis=-1)  # (H,W,3)
        mx = np.max(stacked)
        if mx > 1e-6:
            stacked = stacked / mx
        if float(np.sum(stacked)) <= norm_sum_thr:
            continue
        collected.append(stacked)
        if len(collected) >= slices_per_case:
            break

    if len(collected) == 0:
        out = np.zeros((slices_per_case, img_px_size, img_px_size, 3), dtype=np.float32)
        return out

    while len(collected) < slices_per_case:
        collected.append(collected[-1])

    return np.stack(collected[:slices_per_case], axis=0).astype(np.float32)


def load_dataset_t2(train_dir, labels_df, img_px_size=150, slices_per_case=6):
    X_list, y_list = [], []
    available_cases = sorted([f.name for f in os.scandir(train_dir) if f.is_dir()])
    id_to_label = dict(
        zip(
            labels_df["BraTS21ID"].astype(str).str.zfill(5),
            labels_df["MGMT_value"].astype(int),
        )
    )

    for case_id in available_cases:
        if case_id in BAD_CASES:
            continue
        if case_id not in id_to_label:
            continue
        case_dir = os.path.join(train_dir, case_id)
        x_case = _load_case_t2_slices(
            case_dir, img_px_size=img_px_size, slices_per_case=slices_per_case
        )
        X_list.append(x_case)
        y_list.append(np.full((x_case.shape[0],), id_to_label[case_id], dtype=np.int32))

    if len(X_list) == 0:
        raise RuntimeError("No training data loaded. Check paths and labels.")
    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    return X, y


def load_test_cases_t2(test_dir, img_px_size=150, slices_per_case=6):
    case_ids = sorted(
        [
            f.name
            for f in os.scandir(test_dir)
            if f.is_dir() and f.name.isdigit() and len(f.name) == 5
        ]
    )
    if len(case_ids) == 0:
        raise RuntimeError(f"No test cases found in {test_dir}. Check dataset path.")
    X_cases = []
    for case_id in case_ids:
        case_dir = os.path.join(test_dir, case_id)
        x_case = _load_case_t2_slices(
            case_dir, img_px_size=img_px_size, slices_per_case=slices_per_case
        )
        X_cases.append(x_case)
    X_cases = np.stack(X_cases, axis=0)  # (N_cases, S, H, W, 3)
    return case_ids, X_cases




In [3]:
labels = pd.read_csv(LABELS_CSV)
labels["BraTS21ID"] = labels["BraTS21ID"].astype(str).str.zfill(5)
print(labels.head())
print("Labels shape:", labels.shape)

# If TF isn't available, skip training/data decoding and go straight to fallback submission later.
if TF_AVAILABLE:
    X, y = load_dataset_t2(
        TRAIN_DIR, labels, img_px_size=IMG_PX_SIZE, slices_per_case=SLICES_PER_CASE
    )
    print("Loaded X:", X.shape, "y:", y.shape, "pos_rate:", float(y.mean()))

    idx = np.arange(len(y))
    rng = np.random.RandomState(SEED)
    rng.shuffle(idx)
    split = int(0.85 * len(idx))
    tr_idx, va_idx = idx[:split], idx[split:]

    X_tr, y_tr = X[tr_idx], y[tr_idx]
    X_va, y_va = X[va_idx], y[va_idx]
    print("Train:", X_tr.shape, "Val:", X_va.shape)
else:
    X_tr = y_tr = X_va = y_va = None




  BraTS21ID  MGMT_value
0     00642           0
1     00100           1
2     00309           0
3     00301           0
4     00464           0
Labels shape: (526, 2)


2026-05-13 01:16:23.409334: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Loaded X: (3138, 150, 150, 3) y: (3138,) pos_rate: 0.5200764818355641
Train: (2667, 150, 150, 3) Val: (471, 150, 150, 3)


In [4]:
def build_model(input_shape=(150, 150, 3)):
    inputs = tf.keras.Input(shape=input_shape)
    x = layers.Conv2D(16, 3, padding="same", activation="relu")(inputs)
    x = layers.MaxPool2D()(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(2, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs)

    class AUCPos(tf.keras.metrics.AUC):
        def __init__(self, name="auc", **kwargs):
            super().__init__(name=name, curve="ROC", **kwargs)

        def update_state(self, y_true, y_pred, sample_weight=None):
            y_true = tf.cast(y_true, tf.float32)
            y_pred_pos = y_pred[:, 1]
            return super().update_state(y_true, y_pred_pos, sample_weight=sample_weight)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=[AUCPos(name="auc")],
    )
    return model


if TF_AVAILABLE:
    model_T2 = build_model(input_shape=(IMG_PX_SIZE, IMG_PX_SIZE, 3))
    model_T2.summary()
else:
    model_T2 = None




Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 150, 150, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 75, 75, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 75, 75, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 37, 37, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 37, 37, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,714 (92.63 KB)

 Trainable params: 23,714 (92.63 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
BATCH_SIZE = 32
EPOCHS = 3  # keep as provided

if TF_AVAILABLE:
    history = model_T2.fit(
        X_tr,
        y_tr,
        validation_data=(X_va, y_va),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=2,
    )




Epoch 1/3
84/84 - 8s - 100ms/step - auc: 0.4951 - loss: 0.6929 - val_auc: 0.5000 - val_loss: 0.6927
Epoch 2/3
84/84 - 6s - 75ms/step - auc: 0.4898 - loss: 0.6926 - val_auc: 0.5000 - val_loss: 0.6925
Epoch 3/3
84/84 - 6s - 70ms/step - auc: 0.4944 - loss: 0.6924 - val_auc: 0.5000 - val_loss: 0.6925


In [6]:
if TF_AVAILABLE:
    test_case_ids, X_test_cases = load_test_cases_t2(
        TEST_DIR, img_px_size=IMG_PX_SIZE, slices_per_case=SLICES_PER_CASE
    )
    print("Test cases:", len(test_case_ids), "X_test_cases:", X_test_cases.shape)

    N = X_test_cases.shape[0]
    S = X_test_cases.shape[1]
    X_test_flat = X_test_cases.reshape(N * S, IMG_PX_SIZE, IMG_PX_SIZE, 3)

    preds_flat = model_T2.predict(X_test_flat, batch_size=BATCH_SIZE, verbose=0)
    pred_pos_flat = preds_flat[:, 1].astype(np.float32)

    pred_pos_cases = pred_pos_flat.reshape(N, S).mean(axis=1)
    print(
        "Pred stats:",
        float(pred_pos_cases.min()),
        float(pred_pos_cases.max()),
        float(pred_pos_cases.mean()),
    )
else:
    # --- FIX: End-to-end safety: if TF is unavailable, still produce a valid submission.
    sample = pd.read_csv(SAMPLE_SUB)
    sample["BraTS21ID"] = sample["BraTS21ID"].astype(str).str.zfill(5)
    test_case_ids = sample["BraTS21ID"].tolist()
    pred_pos_cases = np.full((len(test_case_ids),), 0.5, dtype=np.float32)




Test cases: 59 X_test_cases: (59, 6, 150, 150, 3)
Pred stats: 0.5162822008132935 0.5162822008132935 0.5162822008132935


In [7]:
def create_sub(case_ids, pred_pos):
    df = pd.DataFrame(
        {
            "BraTS21ID": [str(x).zfill(5) for x in case_ids],
            "MGMT_value": pred_pos.astype(float),
        }
    )
    return df


sub_df = create_sub(test_case_ids, pred_pos_cases)

sample = pd.read_csv(SAMPLE_SUB)
sample["BraTS21ID"] = sample["BraTS21ID"].astype(str).str.zfill(5)

sub_df = sample[["BraTS21ID"]].merge(sub_df, on="BraTS21ID", how="left")
sub_df["MGMT_value"] = sub_df["MGMT_value"].fillna(float(sub_df["MGMT_value"].mean()))

print(sub_df.head())
print("Submission shape:", sub_df.shape)

sub_df.to_csv("submission.csv", index=False)
print("Wrote submission.csv with columns:", list(sub_df.columns))
print("File size (bytes):", os.path.getsize("submission.csv"))

  BraTS21ID  MGMT_value
0     00356    0.516282
1     00140    0.516282
2     00757    0.516282
3     00275    0.516282
4     00570    0.516282
Submission shape: (59, 2)
Wrote submission.csv with columns: ['BraTS21ID', 'MGMT_value']
File size (bytes): 1496
